[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/12_more_peft.ipynb)

# More PEFT — DoRA, IA³, BitFit, adapters

> **Fine-tuning series — 12 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## 8. More PEFT (how-axis): DoRA · IA³ · BitFit · adapters

The how-axis beyond LoRA. **DoRA** decomposes weights into magnitude+direction and
LoRA-adapts only the direction — usually beats plain LoRA, one flag. **IA³** learns
per-channel scaling vectors (even fewer params than LoRA). **BitFit** trains *only
bias* terms. **Bottleneck adapters** (Houlsby/Pfeiffer) insert small down→up modules
— the original PEFT idea, and they live in the separate `adapters` library, not PEFT.

**DoRA** — LoRA with `use_dora=True` (the only change vs. §2).

In [ ]:
dora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    use_dora=True,                                  # <-- DoRA
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
trainer = SFTTrainer(model=model, train_dataset=sft_ds, peft_config=dora_cfg,
    args=SFTConfig(output_dir="ft_dora", dataset_text_field="text",
                   per_device_train_batch_size=2, max_steps=20, learning_rate=2e-4,
                   logging_steps=5, bf16=BF16_OK, report_to="none"))
trainer.train(); del trainer, model; cleanup()
# Cousins, all LoraConfig flags: use_rslora=True (rank-stabilized), init_lora_weights="pissa"/"eva"/"loftq".

**IA³** — learned scaling vectors; `feedforward_modules` must be a subset of `target_modules`.

In [ ]:
from peft import IA3Config, get_peft_model, TaskType
ia3_cfg = IA3Config(task_type=TaskType.CAUSAL_LM,
    target_modules=["k_proj", "v_proj", "down_proj"],
    feedforward_modules=["down_proj"])
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
model = get_peft_model(model, ia3_cfg)
model.print_trainable_parameters()                  # even tinier than LoRA
trainer = SFTTrainer(model=model, train_dataset=sft_ds,
    args=SFTConfig(output_dir="ft_ia3", dataset_text_field="text",
                   per_device_train_batch_size=2, max_steps=20, learning_rate=3e-3,
                   logging_steps=5, bf16=BF16_OK, report_to="none"))
trainer.train(); del trainer, model; cleanup()

**BitFit** — train only bias terms. No PEFT config; just freeze everything else.
Footprint depends on the architecture *having* biases — modern LLMs (Qwen/Llama)
have few, so this is more of a baseline than a go-to.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
for name, p in model.named_parameters():
    p.requires_grad = name.endswith(".bias") or name.endswith("bias")
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable bias params:", n_train)
trainer = SFTTrainer(model=model, train_dataset=sft_ds,
    args=SFTConfig(output_dir="ft_bitfit", dataset_text_field="text",
                   per_device_train_batch_size=2, max_steps=20, learning_rate=1e-4,
                   logging_steps=5, bf16=BF16_OK, report_to="none"))
trainer.train(); del trainer, model; cleanup()

**Bottleneck adapters** — the classic Houlsby/Pfeiffer modules, via the
`adapters` library (AdapterHub), *not* PEFT.

In [ ]:
# pip install adapters
import adapters
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
adapters.init(model)                                # enable adapter support
model.add_adapter("bottleneck", config="seq_bn")    # Pfeiffer; "double_seq_bn" = Houlsby
model.train_adapter("bottleneck")                   # freeze base, train only the adapter
print(model.adapter_summary())
# Train with adapters' own AdapterTrainer (HF-Trainer compatible) or a plain loop.
# config="seq_bn[reduction_factor=16]" controls the bottleneck size.
del model; cleanup()

### More LoRA variants (quick reference)

All of these are small twists on LoRA — same idea, different initialization or scaling. Most are
one flag on `LoraConfig`; two have their own config class in PEFT.

| Variant | What's different | How to enable |
|---|---|---|
| **rsLoRA** | rank-stabilized scaling (`α/√r` not `α/r`) — lets high ranks actually help | `LoraConfig(use_rslora=True)` |
| **PiSSA** | init adapters from the SVD of the weights (not zeros) → faster convergence | `LoraConfig(init_lora_weights="pissa")` |
| **LoftQ** | quantization-aware init that offsets 4-bit error — better QLoRA starts | `LoraConfig(init_lora_weights="loftq", loftq_config=LoftQConfig(loftq_bits=4))` |
| **AdaLoRA** | budgets rank *adaptively* across layers, pruning unimportant directions | `from peft import AdaLoraConfig` |
| **VeRA** | freezes shared random A/B, trains only tiny per-layer scaling vectors → fewest params | `from peft import VeraConfig` |

In [ ]:
from peft import LoraConfig
# the one-flag variants are just a different LoraConfig:
rslora_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj","v_proj"], use_rslora=True)
pissa_cfg  = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj","v_proj"],
                        init_lora_weights="pissa")
# AdaLoRA / VeRA use dedicated configs:
# from peft import AdaLoraConfig, VeraConfig
print("rsLoRA use_rslora:", rslora_cfg.use_rslora, "| PiSSA init:", pissa_cfg.init_lora_weights)